# Day 15 — LLM Inference I
## Autoregressive Decoding → Prefill / Decode → KV Cache → Latency / Throughput

今天开始 Week 3。

主资源：**CS336 Lecture 10: Inference**。

今天不追求把 Lecture 10 全部吃完，而是先抓住最核心的一条主线：

```text
Training
↓
parallel over sequence

Inference
↓
autoregressive generation
↓
Prefill
↓
KV Cache
↓
Decode one token at a time
↓
Latency / Throughput / Memory-bound
```

Lecture 10 把 inference 的核心区别讲得很明确：

- Training：一次可以看到整段 token，因此 sequence 维可以并行。
- Inference：生成是 sequential 的，不能沿生成时间维完全并行。
- Naive inference 会重复计算历史 prefix。
- KV cache 用来复用历史 token 的 K/V。
- Inference 分成 **prefill** 与 **generation / decode** 两阶段。
- Prefill 更容易 compute-bound；generation 往往更 memory-bound。

今天重点对应 Lecture 10 中：
- `Understanding the inference workload`
- `review_transformer`
- `review_of_arithmetic_intensity`
- `arithmetic_intensity_of_inference`
- `throughput_and_latency`

今天先不深挖：
- quantization
- pruning
- speculative sampling
- continuous batching
- PagedAttention

这些后面再来。

# 0. 今天最重要的起点问题

## 必须回答 1

训练 Transformer 时，我们可以一次把整个 sequence 喂进去：

```text
[x1, x2, x3, x4]
```

并并行得到各位置的 logits。

但生成时却通常是：

```text
prompt
↓
生成 token 1
↓
把 token 1 接回上下文
↓
生成 token 2
↓
...
```

为什么生成不能像训练一样，把“未来要生成的所有 token”一次性并行算完？

你的回答：训练时，完整的目标序列已经已知，因此虽然使用 causal mask 保证第 t 个位置只能依赖前面的 token，但所有位置仍然可以在一次 forward 中并行计算。
生成时，未来 token 还没有产生，而第 t+1 个 token 的概率依赖已经生成的第 t 个 token，因此必须先生成当前 token，再把它加入上下文后继续生成下一个 token。
所以 autoregressive generation 本质上存在沿时间维的顺序依赖，不能像训练一样把未来所有 token 一次性并行生成。

# 1. Autoregressive generation

语言模型建模的是：

$$
p(x_1, x_2, \dots, x_T)
=
\prod_{t=1}^{T}
p(x_t \mid x_{<t})
$$

也就是说，第 $t$ 个 token 的概率依赖前面已经存在的 token。

训练时，完整目标序列已经知道，所以我们可以用 causal mask 一次性并行计算所有位置。

生成时，未来 token 还不存在，所以：

$x_{t+1}$ 依赖 $x_t$

必须先生成前一个，再生成后一个。

In [3]:
# 一个极简 autoregressive sampling 演示
# 这里不是 Transformer，只是用一个 toy transition rule 模拟：
# “下一个 token 的分布依赖当前 prefix”。

import torch
import torch.nn.functional as F

vocab = ["A", "B", "C", "<END>"]

transition_logits = {
    "A": torch.tensor([0.2, 2.0, 0.5, -1.0]),
    "B": torch.tensor([0.1, 0.4, 2.2, -0.5]),
    "C": torch.tensor([0.5, 0.2, 0.3, 1.8]),
}

def sample_next(current_token, temperature=1.0):
    logits = transition_logits[current_token] / temperature
    probs = F.softmax(logits, dim=0)
    idx = torch.multinomial(probs, num_samples=1).item()
    return vocab[idx], probs

current = "A"
sequence = [current]

for step in range(10):
    next_token, probs = sample_next(current, temperature=1.0)
    print(
        f"step={step:02d} | current={current} | "
        f"probs={[round(float(x), 3) for x in probs]}"
    )
    sequence.append(next_token)

    if next_token == "<END>":
        break

    current = next_token

print("generated:", sequence)

step=00 | current=A | probs=[0.115, 0.695, 0.155, 0.035]
step=01 | current=B | probs=[0.09, 0.122, 0.738, 0.05]
step=02 | current=C | probs=[0.161, 0.119, 0.131, 0.589]
generated: ['A', 'B', 'C', '<END>']


### 必须回答 2

上面这个 toy 例子虽然不是 Transformer，但它体现了 autoregressive generation 的什么本质？

提示：

> 为什么第 3 个 token 不能在第 1 个 token 还没生成时直接确定？

你的回答：因为 autoregressive generation 中，第 t 个 token 的概率分布依赖此前已经生成的 token。在第 2 个 token 尚未确定时，第 3 个 token 所依赖的上下文也尚未确定，因此无法提前确定第 3 个 token，只能先生成第 2 个 token，再继续生成第 3 个 token。

---

# 2. Temperature / sampling

Lecture 10 的重点主要在 inference efficiency，但生成阶段最基本的 sampling 还是要会。

给定 logits：

$$
z_i
$$

temperature sampling 使用：

$$
p_i
=
\mathrm{softmax}
\left(
\frac{z_i}{T}
\right)
$$

其中：

- $T < 1$：分布更尖锐，更 deterministic
- $T = 1$：原分布
- $T > 1$：分布更平，更随机

In [2]:
logits = torch.tensor([3.0, 2.0, 1.0, 0.0])

for T in [0.5, 1.0, 2.0]:
    probs = F.softmax(logits / T, dim=0)
    print(
        f"T={T}:",
        [round(float(x), 4) for x in probs]
    )

T=0.5: [0.865, 0.1171, 0.0158, 0.0021]
T=1.0: [0.6439, 0.2369, 0.0871, 0.0321]
T=2.0: [0.4551, 0.276, 0.1674, 0.1015]


### 必须回答 3

为什么：

```text
temperature 越低
→ 最大 logit 对应 token 的概率越高
```

而 temperature 越高，分布会更平？

不要只写结论，结合：

$$
\mathrm{softmax}(z/T)
$$

解释。

你的回答：当 T<1 时，相当于将 logits 放大，使不同 logits 之间的差距增大，因此 softmax 后概率分布更加尖锐，最大 logit 对应 token 的概率更高。

当 T>1 时，相当于将 logits 缩小，使 logits 之间的差距减小，因此 softmax 后概率分布更加平坦，低概率 token 被采样到的可能性增加。

# 3. Top-k sampling

Top-k 的思想：

> 只保留概率 / logit 最大的 k 个 token，其他 token 不允许被采样。

例如：

```text
原 vocabulary:
[A, B, C, D, E]

top-k = 2
↓
只保留最可能的两个 token
```

In [3]:
def top_k_filter(logits, k):
    values, indices = torch.topk(logits, k)

    filtered = torch.full_like(logits, float("-inf"))
    filtered[indices] = logits[indices]

    return filtered

logits = torch.tensor([3.0, 2.0, 1.0, 0.5, -0.2])

filtered = top_k_filter(logits, k=2)

print("original logits:", logits)
print("filtered logits:", filtered)
print("probabilities:", F.softmax(filtered, dim=0))

original logits: tensor([ 3.0000,  2.0000,  1.0000,  0.5000, -0.2000])
filtered logits: tensor([3., 2., -inf, -inf, -inf])
probabilities: tensor([0.7311, 0.2689, 0.0000, 0.0000, 0.0000])


### 必须回答 4

为什么被过滤掉的 logits 要设成：

```python
-inf
```

而不是设成：

```python
0
```

提示：softmax。

你的回答：Top-k 将被过滤 token 的 logit 设置为 $-\infty$，因为 softmax 中 $e^{-\infty}=0$，因此这些 token 的最终概率严格为 0，不可能被采样。如果设置为 0，则 $e^0=1$，这些 token 仍然会获得非零概率

# 4. Top-p / nucleus sampling

Top-p 不固定保留多少个 token。

它按概率从大到小排序，然后保留最小的一组 token，使累计概率达到阈值 $p$。

例如：

```text
probabilities:
0.50
0.25
0.15
0.06
0.04
```

如果：

```text
top-p = 0.80
```

那么可能保留：

```text
0.50 + 0.25 + 0.15 = 0.90
```

也就是前三个 token。

In [4]:
def top_p_filter(logits, p=0.9):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=0)

    cumulative = torch.cumsum(sorted_probs, dim=0)

    # 保留第一个使累计概率越过 p 的 token
    remove = cumulative > p
    remove[1:] = remove[:-1].clone()
    remove[0] = False

    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))

    filtered = torch.full_like(logits, float("-inf"))
    filtered[sorted_indices] = sorted_logits

    return filtered

logits = torch.tensor([3.0, 2.0, 1.0, 0.5, -0.2])

filtered = top_p_filter(logits, p=0.8)

print("original probs:", F.softmax(logits, dim=0))
print("top-p probs:", F.softmax(filtered, dim=0))

original probs: tensor([0.6150, 0.2262, 0.0832, 0.0505, 0.0251])
top-p probs: tensor([0.7311, 0.2689, 0.0000, 0.0000, 0.0000])


### 必须回答 5

Top-k 和 Top-p 的区别是什么？

为什么说：

```text
top-k:
候选 token 数固定

top-p:
候选 token 数动态变化
```

你的回答：Top-k 固定候选 token 的数量 k，只保留概率最高的 k 个 token；Top-p 不固定候选数量，而是将 token 按概率从高到低排序，保留最小的一个候选集合，使其累计概率达到或超过 p。因此 Top-p 的候选 token 数量会根据当前概率分布动态变化。

---

# 5. Naive inference：为什么会重复算？

CS336 Lecture 10 给出的 naive inference 直觉：

> 为了生成每一个新 token，都把整个历史 prefix 重新喂进 Transformer。

例如 prompt：

```text
A B C
```

生成：

```text
D
```

下一步 naive 做：

```text
A B C D
```

再完整 forward。

下一步：

```text
A B C D E
```

又完整 forward。

问题：

> 前面 A/B/C 对应的大量计算明明已经做过了，却又被重复计算。

## 必须回答 6

为什么 naive autoregressive inference 会产生大量重复计算？

请用：

```text
A B C
→ D
→ E
```

这个例子解释。

你的回答：Naive autoregressive inference 在每生成一个新 token 时，都会把完整历史序列重新输入 Transformer。例如先用 A B C 生成 D，随后为了生成 E，又将 A B C D 整体重新计算。因此 A、B、C 对应的历史 K/V 和其他中间计算会被重复执行。随着生成序列变长，这种重复计算越来越严重。KV cache 的作用就是保存历史 token 已经计算好的 K/V，使下一步只需计算新 token 的 K/V 并追加到缓存，而不必重新计算所有历史 token。

# 6. KV Cache

Transformer attention 中：

$$
Q = XW_Q,\qquad
K = XW_K,\qquad
V = XW_V
$$

生成下一个 token 时：

- 新 token 的 Query 是新的；
- 历史 token 的 Key / Value 并没有变化。

所以可以把历史 token 的 $K,V$ 保存起来。

这就是：

# KV Cache

CS336 Lecture 10 的核心观察：

> a lot of work can be shared across prefixes

并明确把 KV cache 放在 HBM 中保存。

In [5]:
# Toy KV-cache shape demo
#
# 假设：
# batch = 2
# sequence_length = 5
# num_kv_heads = 4
# head_dim = 8

B = 2
S = 5
K = 4
H = 8

key_cache = torch.randn(B, S, K, H)
value_cache = torch.randn(B, S, K, H)

print("K cache shape:", key_cache.shape)
print("V cache shape:", value_cache.shape)

K cache shape: torch.Size([2, 5, 4, 8])
V cache shape: torch.Size([2, 5, 4, 8])


Lecture 10 使用的 KV cache 维度思想是：

```text
sequence
× token
× layer
× KV head
× head dimension
```

并且每个 token 要同时存 Key 和 Value。

---

## 必须回答 7

为什么 KV cache 保存的是历史 token 的 **K 和 V**，而不是直接把过去所有 attention output 保存下来就够了？

提示：

新 token 的 $Q$ 需要和历史的什么东西做 attention？

你的回答：KV cache 保存历史 token 的 K 和 V，是因为每个新 token 都会产生新的 Query，而新的 Query 需要重新与所有历史 Key 计算 attention score，并据此对历史 Value 做新的加权求和。历史 attention output 只是旧 Query 对历史 V 的一次特定加权结果，已经丢失了各个历史 token 的独立 K/V 信息，因此不能用于新的 Query 重新计算 attention。

# 7. Prefill vs Decode / Generation

Lecture 10 把 inference 分成两个阶段：

## Prefill

给定完整 prompt：

```text
[user prompt tokens]
```

一次处理整个 prompt。

特点：

- prompt 已经全部知道；
- token 之间可以并行；
- 更像 training forward；
- 通常更容易 compute-bound；
- TTFT 很大程度由 prefill 决定。

## Decode / Generation

每次只生成一个新 token：

```text
token_t
→ token_{t+1}
```

特点：

- sequential；
- 每步 $T=1$；
- 需要读取模型参数 + KV cache；
- 常常 memory-bound。

### 必须回答 8

解释：

```text
Prefill:
T = S

Generation:
T = 1
```

这里的直觉是什么？

你的回答：Prefill 时，整个 prompt 已经已知，因此所有 prompt token 可以通过 causal mask 并行处理，通常形成较大的矩阵计算，arithmetic intensity 较高，因此常常更接近 compute-bound。
Generation 时，每一步未来 token 都依赖前一步实际生成的 token，因此时间维上必须顺序执行；每步通常只有 T=1，但仍需读取模型权重和不断增长的历史 KV cache，因此 arithmetic intensity 较低，常常 memory-bound。

# 8. 一个极简 Prefill / Decode shape 实验

下面不实现完整 Transformer，只模拟：

```text
Prefill:
一次处理 S 个 token

Decode:
每次只处理 1 个新 token
```

In [6]:
B = 2
S = 6
D = 16

# Prefill:
x_prefill = torch.randn(B, S, D)

# Decode:
x_decode = torch.randn(B, 1, D)

print("prefill input shape:", x_prefill.shape)
print("decode input shape :", x_decode.shape)

prefill input shape: torch.Size([2, 6, 16])
decode input shape : torch.Size([2, 1, 16])


### 必须回答 9

为什么这两个 shape：

```text
Prefill: [B, S, D]
Decode : [B, 1, D]
```

会导致硬件利用率非常不同？

先不需要讲精确 FLOPs，只从：

```text
一次能做多少 token 的矩阵计算
```

这个角度解释。

你的回答：Prefill 会把 S 个 token 一起组成一个大矩阵参与一次 batched / matrix-matrix computation；Decode 每个 request 每步通常只有 1 个新 token，所以对应的是更“薄”的矩阵计算。

---

# 9. Arithmetic Intensity

CS336 Lecture 10 先复习 arithmetic intensity：

$$
\text{Arithmetic Intensity}
=
\frac{\text{FLOPs}}
{\text{Bytes transferred}}
$$

直觉：

```text
每从 HBM 搬 1 byte 数据
到底能做多少计算？
```

如果：

```text
arithmetic intensity 很高
```

说明搬一次数据能做很多计算，更可能是 compute-bound。

如果：

```text
arithmetic intensity 很低
```

说明大量时间在搬数据，更可能是 memory-bound。

Lecture 10 对一个简单矩阵乘法：

$$
X[B,D]W[D,F]
$$

推导出在某些常见假设下，arithmetic intensity 大约随 batch size $B$ 增大。

极端情况：

```text
B = 1
```

接近 matrix-vector multiply：

```text
读巨大 W
↓
只为了算一个 token / 一个 vector
```

所以 arithmetic intensity 很低。

Lecture 10 明确说：

> This is basically what happens with inference.

In [7]:
# 一个纯直觉实验：
# 假设一组模型权重需要读取 100 单位内存。
# 不同 batch 可以复用同一份权重。

weight_read = 100.0

for B in [1, 2, 8, 32, 128]:
    useful_compute = B
    rough_intensity = useful_compute / weight_read

    print(
        f"B={B:3d} | "
        f"rough compute/read ratio={rough_intensity:.3f}"
    )

B=  1 | rough compute/read ratio=0.010
B=  2 | rough compute/read ratio=0.020
B=  8 | rough compute/read ratio=0.080
B= 32 | rough compute/read ratio=0.320
B=128 | rough compute/read ratio=1.280


### 必须回答 10

为什么 batch size 增大时，MLP 层的 inference throughput 往往可以提高？

用一句核心话回答：

> 同一份权重被 ________。

你的回答：同一份权重被更多 token / request 复用

# 10. Attention 的 Prefill vs Generation

Lecture 10 对 attention 的 arithmetic intensity 推导：

$$
\text{intensity}
=
\frac{ST}{S+T}
$$

其中：

- $S$：历史 / conditioning tokens
- $T$：本次要计算 logits 的 token 数

Prefill：

$$
T=S
$$

所以：

$$
\text{intensity}
=
\frac{S}{2}
$$

Generation：

$$
T=1
$$

所以：

$$
\text{intensity}
=
\frac{S}{S+1}
<1
$$

这就是 Lecture 10 非常关键的结论之一：

> **Prefill 更容易 compute-bound，而 generation attention 极度 memory-bound。**

In [8]:
for S in [16, 128, 1024, 8192]:
    prefill_intensity = S / 2
    generation_intensity = S / (S + 1)

    print(
        f"S={S:5d} | "
        f"prefill={prefill_intensity:8.2f} | "
        f"generation={generation_intensity:.4f}"
    )

S=   16 | prefill=    8.00 | generation=0.9412
S=  128 | prefill=   64.00 | generation=0.9922
S= 1024 | prefill=  512.00 | generation=0.9990
S= 8192 | prefill= 4096.00 | generation=0.9999


### 必须回答 11

为什么：

```text
S 很大
```

并不会让 generation attention 的 arithmetic intensity 变得很高？

观察：

$$
\frac{S}{S+1}
$$

当 $S\to\infty$ 时趋近多少？

你的回答：趋近于1， 所以即使S很大，generation attention 的 arithmetic intensity 的上线也就是1，无法更大

---

# 11. 为什么 attention batching 没 MLP 那么有效？

Lecture 10 讲了一个很值得记住的区别。

MLP：

```text
不同 sequence
↓
都使用同一组 W_up / W_gate / W_down
```

所以 batch 增大，可以让很多请求共同摊薄读取模型参数的成本。

Attention：

```text
sequence 1 → 自己的 KV cache
sequence 2 → 自己的 KV cache
sequence 3 → 自己的 KV cache
```

每个 sequence 都有自己的 KV cache。

所以增加 batch 并不能像 MLP 那样有效提高 attention 的 arithmetic intensity。

### 必须回答 12

为什么 Lecture 10 会说：

> MLP batching 可以 amortize weight reads，但 attention 的 KV cache 是 per-sequence 的？

请用自己的话解释。

你的回答： MLP 的模型权重对所有 sequence 是共享的，因此增大 batch 后，同一份权重可以被更多 token 复用，从而摊薄 weight read 的成本。Attention 的 KV cache 则由每条 sequence 自己的历史 token 计算得到；不同 sequence 的上下文不同，因此 K/V 也不同，每条 sequence 必须读取自己的 KV cache。所以增大 batch 时，KV cache 的读取量也近似随 batch 一起增加，无法像 MLP 权重那样有效 amortize memory read。

# 12. Latency / Throughput / TTFT

Lecture 10 定义了三个 inference 指标：

## TTFT — Time To First Token

从请求到第一个生成 token 出现的时间。

主要受：

```text
prefill
```

影响。

## Latency

一个请求生成 token 的速度，常看：

```text
seconds / token
```

## Throughput

整个系统单位时间一共生成多少 token：

```text
tokens / second
```

通常：

```text
更大 batch
→ throughput 更高
→ latency 可能更差
```

这就是 serving 中非常重要的 trade-off。

In [9]:
# 一个 toy latency-throughput tradeoff
#
# 假设：
# fixed model read cost = 1.0
# per-request KV cost = 0.02 * B
#
# latency ~ fixed + kv cost
# throughput ~ B / latency

for B in [1, 4, 16, 64, 128]:
    latency = 1.0 + 0.02 * B
    throughput = B / latency

    print(
        f"B={B:3d} | "
        f"latency={latency:.3f} | "
        f"throughput={throughput:.2f}"
    )

B=  1 | latency=1.020 | throughput=0.98
B=  4 | latency=1.080 | throughput=3.70
B= 16 | latency=1.320 | throughput=12.12
B= 64 | latency=2.280 | throughput=28.07
B=128 | latency=3.560 | throughput=35.96


### 必须回答 13

为什么：

```text
larger batch
```

可能同时带来：

```text
更高 throughput
但更差 latency
```

这两个结论并不矛盾？

你的回答：Larger batch 可以让更多请求同时参与计算，提高模型权重的复用率和硬件利用率，因此系统单位时间处理的 token 数增加，throughput 往往提高。但更大的 batch 也意味着每轮需要处理更多数据和 KV cache，因此单轮 latency 可能增加。这两者并不矛盾，因为 throughput 衡量系统整体处理能力，而 latency 衡量单个请求/生成步骤需要等待的时间。

---

# 13. KV Cache Memory 估算

Lecture 10 给出的 KV cache per sequence 直觉：

$$
S \times (K\cdot H)\times L\times 2\times \text{bytes}
$$

其中：

- $S$：sequence length
- $K$：KV heads
- $H$：head dim
- $L$：layers
- 第一个 $2$：Key + Value
- bytes：例如 bf16 = 2 bytes

In [10]:
def kv_cache_bytes(
    seq_len,
    num_kv_heads,
    head_dim,
    num_layers,
    bytes_per_value=2
):
    return (
        seq_len
        * num_kv_heads
        * head_dim
        * num_layers
        * 2
        * bytes_per_value
    )

configs = [
    ("small", 1024, 8, 128, 24),
    ("medium", 4096, 8, 128, 32),
    ("long-context", 32768, 8, 128, 32),
]

for name, S, K, H, L in configs:
    memory = kv_cache_bytes(S, K, H, L)

    print(
        f"{name:12s} | "
        f"{memory / 1024**2:.2f} MiB per sequence"
    )

small        | 96.00 MiB per sequence
medium       | 512.00 MiB per sequence
long-context | 4096.00 MiB per sequence


### 必须回答 14

根据公式解释：

为什么 context length 变长，会直接让 KV cache memory 增长？

这是：

```text
O(1)
O(log S)
O(S)
O(S^2)
```

中的哪一种？

你的回答：O(S)

# 14. GQA 为什么能省 KV Cache？

Lecture 10 接下来用 GQA 作为减少 KV cache 的第一个例子。

设：

```text
N = query heads
K = key/value heads
```

传统 MHA：

```text
K = N
```

MQA：

```text
K = 1
```

GQA：

```text
1 < K < N
```

Lecture 10 的关键结论：

> GQA 通过减少 KV heads，把 KV cache 大约缩小 $N/K$ 倍。

In [1]:
N = 40

for K in [40, 20, 8, 4, 1]:
    reduction = N / K

    print(
        f"query_heads={N:2d} | "
        f"kv_heads={K:2d} | "
        f"KV-cache reduction ≈ {reduction:.1f}x"
    )

query_heads=40 | kv_heads=40 | KV-cache reduction ≈ 1.0x
query_heads=40 | kv_heads=20 | KV-cache reduction ≈ 2.0x
query_heads=40 | kv_heads= 8 | KV-cache reduction ≈ 5.0x
query_heads=40 | kv_heads= 4 | KV-cache reduction ≈ 10.0x
query_heads=40 | kv_heads= 1 | KV-cache reduction ≈ 40.0x


### 必须回答 15

为什么减少 KV heads 会改善 inference latency / throughput？

请沿着：

```text
KV cache 变小
↓
需要从 HBM 搬的数据减少
↓
...
```

继续解释。

你的回答：减少 KV heads 后，多个 Query heads 可以共享同一组 K/V，从而减少每个 token 需要缓存的 K/V 数量，使 KV cache 变小。Decode 阶段需要从 HBM 读取的 KV cache 数据随之减少，因此降低 memory traffic。由于 generation attention 通常是 memory-bound，这可以降低 inference latency，并提高 throughput

---

# 15. Day 15 小实验：把整条 inference pipeline 串起来

我们现在用一个非常简化的 simulation 来表示：

```text
Prompt
↓
Prefill
↓
创建 KV cache
↓
Decode token
↓
append KV cache
↓
Decode token
...
```

这个 simulation 不实现真实 attention，只练 inference control flow 和 shape。

In [4]:
B = 1
prompt_len = 4
num_layers = 2
num_kv_heads = 2
head_dim = 4
num_new_tokens = 5

# Prefill 后的 KV cache
K_cache = torch.randn(
    num_layers,
    B,
    prompt_len,
    num_kv_heads,
    head_dim
)

V_cache = torch.randn(
    num_layers,
    B,
    prompt_len,
    num_kv_heads,
    head_dim
)

print("after prefill:")
print("K:", K_cache.shape)
print("V:", V_cache.shape)

for step in range(num_new_tokens):
    # 模拟当前 decode token 产生的新 K/V
    new_K = torch.randn(
        num_layers,
        B,
        1,
        num_kv_heads,
        head_dim
    )

    new_V = torch.randn(
        num_layers,
        B,
        1,
        num_kv_heads,
        head_dim
    )

    # append 到 sequence dimension
    K_cache = torch.cat([K_cache, new_K], dim=2)
    V_cache = torch.cat([V_cache, new_V], dim=2)

    print(
        f"decode step {step + 1}: "
        f"cache sequence length = {K_cache.shape[2]}"
    )

after prefill:
K: torch.Size([2, 1, 4, 2, 4])
V: torch.Size([2, 1, 4, 2, 4])
decode step 1: cache sequence length = 5
decode step 2: cache sequence length = 6
decode step 3: cache sequence length = 7
decode step 4: cache sequence length = 8
decode step 5: cache sequence length = 9


### 必须回答 16

上面 simulation 中：

```python
torch.cat([K_cache, new_K], dim=2)
```

为什么沿：

```text
dim=2
```

拼接？

请根据 shape：

```text
[L, B, S, K, H]
```

解释。

你的回答：KV cache 的 shape 为 [L, B, S, K, H]，其中 dim=2 对应 sequence length \(S\)。每次 decode 一个新 token，都会产生该 token 对应的新 \(K,V\)。因此沿 dim=2 将 new_K/new_V 拼接到历史 KV cache，相当于在 sequence 维度增加一个 token

# 16. Day 15 最终复盘

请不翻前面答案，直接回答。

### 1.
为什么 training 可以 sequence-parallel，而 autoregressive generation 不能完全 parallel？

### 2.
temperature 在 sampling 中控制什么？

### 3.
top-k 和 top-p 的核心区别是什么？

### 4.
naive inference 为什么浪费大量重复计算？

### 5.
KV cache 到底缓存什么？为什么？

### 6.
Prefill 和 Decode 最核心的区别是什么？

### 7.
为什么 generation 更容易 memory-bound？

### 8.
什么叫 arithmetic intensity？

### 9.
为什么更大的 batch 往往提高 throughput？

### 10.
为什么更大的 batch 可能恶化 latency？

### 11.
TTFT、latency、throughput 分别衡量什么？

### 12.
为什么 context length 越大，KV cache 越大？

### 13.
为什么 GQA 能减少 KV cache？

### 14.
用一句话总结：

> **LLM inference 最难的地方不是“模型不会算”，而是 ________。**

你的回答：

# Day 15 完成标准

如果你已经能独立解释下面这条链：

```text
Autoregressive generation
↓
future token unknown
↓
decode sequentially
↓
naive recomputation is wasteful
↓
cache historical K/V
↓
Prefill + Decode
↓
Decode 每次只处理一个新 token
↓
low arithmetic intensity
↓
memory-bound
↓
latency / throughput trade-off
```

那么 Day 15 就完成了。

---

## 明天 Day 16

明天会继续沿 CS336 inference/system 主线：

```text
KV cache reduction
↓
GQA / MLA / local attention
↓
speculative decoding
↓
continuous batching
↓
PagedAttention / vLLM
```

并逐渐把你之前学过的：

```text
FlashAttention
GPU memory / IO
serving
```

重新串回 LLM inference。